<a href="https://colab.research.google.com/github/Sushanth-xy/Pose_estimator_and_action_classifier/blob/Master/updatedtraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Installing opencv-python...")
!pip install opencv-python
print("✅ opencv-python installed successfully.")

Installing opencv-python...
✅ opencv-python installed successfully.


In [ ]:
import os
import json
from google.colab import userdata

# 1. Setup Kaggle Credentials safely
# Make sure you've added KAGGLE_USERNAME and KAGGLE_KEY to the "Keys" (Secrets) tab on the left!
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# Create the hidden folder Kaggle expects
!mkdir -p ~/.kaggle
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": os.environ['KAGGLE_USERNAME'], "key": os.environ['KAGGLE_KEY']}, f)
!chmod 600 ~/.kaggle/kaggle.json

# 2. Install necessary libraries
!pip install -q ultralytics

# 3. Download and Unzip (Your specific commands)
print("Downloading dataset...")
!kaggle datasets download -d sathyakirants/pose-estimation-videos -p ./data
!unzip -q ./data/pose-estimation-videos.zip -d ./data

print("\n✅ Setup Complete! Check the 'data' folder in the sidebar to see your video folders.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.6 MB/s eta 0:00:00
Dataset URL: https://www.kaggle.com/datasets/sathyakirants/pose-estimation-videos
License(s): unknown
100% 6.49G/6.49G [00:36<00:00, 188MB/s]


✅ Setup Complete! Check the 'data' folder in the sidebar to see your video folders.


In [4]:
import cv2
import numpy as np
import torch
from ultralytics import YOLO
from pathlib import Path
from tqdm.auto import tqdm
import os

# 1. Install dependencies
!pip install -q ultralytics

def calculate_angle(a, b, c):
    """Calculates the angle at point B given points A, B, and C."""
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)
    if angle > 180.0: angle = 360 - angle
    return angle

# 2. Configuration
device = 0 if torch.cuda.is_available() else 'cpu'
model = YOLO('yolov8n-pose.pt')
DATA_DIR = Path('./data/UCF-101')
SEQ_LEN = 60
target_actions = ['BodyWeightSquats', 'PushUps', 'PullUps', 'Lunges', 'BenchPress', 'JumpingJack']

X_data = []
y_labels = []

print(f"Processing target actions: {target_actions}")

for idx, action in enumerate(target_actions):
    action_path = DATA_DIR / action
    if not action_path.exists():
        print(f"☀️ Folder {action} not found. Skipping...")
        continue

    video_paths = list(action_path.glob('*.avi')) + list(action_path.glob('*.mp4'))
    print(f"\nἱ4 Processing {action} ({len(video_paths)} videos)")

    for v_path in tqdm(video_paths[:40]):
        cap = cv2.VideoCapture(str(v_path))
        temp_features = []

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            results = model(frame, verbose=False, device=device)

            if results[0].keypoints is not None and len(results[0].keypoints.xy) > 0:
                kp = results[0].keypoints.xy[0].cpu().numpy()
                mid_hip = (kp[11] + kp[12]) / 2
                norm_kp = kp - mid_hip

                l_elbow = calculate_angle(kp[5], kp[7], kp[9])
                r_elbow = calculate_angle(kp[6], kp[8], kp[10])
                l_knee = calculate_angle(kp[11], kp[13], kp[15])
                r_knee = calculate_angle(kp[12], kp[14], kp[16])

                feat = np.concatenate([norm_kp.flatten(), [l_elbow, r_elbow, l_knee, r_knee]])
                temp_features.append(feat)
        cap.release()

        # Sampling: every 5th frame then interpolate to 60
        filtered_5th = temp_features[::5]
        if len(filtered_5th) > 10:
            indices = np.linspace(0, len(filtered_5th) - 1, SEQ_LEN).astype(int)
            sampled_seq = [filtered_5th[i] for i in indices]
            X_data.append(sampled_seq)
            y_labels.append(idx)

# 3. Save processed data
np.save('X_train_enhanced.npy', np.array(X_data))
np.save('y_train_enhanced.npy', np.array(y_labels))
np.save('action_names_filtered.npy', np.array(target_actions))

print(f"\n✅ Extraction Complete! Saved as 'X_train_enhanced.npy'. Shape: {np.array(X_data).shape}")

Processing target actions: ['BodyWeightSquats', 'PushUps', 'PullUps', 'Lunges', 'BenchPress', 'JumpingJack']

ἱ4 Processing BodyWeightSquats (112 videos)


  0%|          | 0/40 [00:00<?, ?it/s]


ἱ4 Processing PushUps (102 videos)


  0%|          | 0/40 [00:00<?, ?it/s]


ἱ4 Processing PullUps (100 videos)


...


ἱ4 Processing BenchPress (160 videos)


  0%|          | 0/40 [00:00<?, ?it/s]


ἱ4 Processing JumpingJack (123 videos)


  0%|          | 0/40 [00:00<?, ?it/s]


✅ Extraction Complete! Saved as 'X_train_enhanced.npy'. Shape: (215, 60, 38)


Remember to transfer both `pose_lstm_model.h5` and `action_names_filtered.npy` to your local machine.

In [6]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# 1. Load the preprocessed data
X = np.load('X_train_enhanced.npy')
y = np.load('y_train_enhanced.npy')
action_names = np.load('action_names_filtered.npy')

# 2. Prepare data for Training
# X shape is (samples, 60, 38) -> 17 keypoints * 2 (x,y) + 4 angles
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# One-hot encode labels
y_train_cat = to_categorical(y_train)
y_val_cat = to_categorical(y_val)

# 3. Define LSTM Model
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X.shape[1], X.shape[2])),
    Dropout(0.2),
    BatchNormalization(),

    LSTM(128, return_sequences=False),
    Dropout(0.2),
    BatchNormalization(),

    Dense(64, activation='relu'),
    Dense(len(action_names), activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# 4. Train the model
print("\nStarting Training...")
history = model.fit(
    X_train, y_train_cat,
    epochs=50,
    batch_size=32,
    validation_data=(X_val, y_val_cat),
    verbose=1
)

# 5. Save the model
model.save('pose_lstm_model.h5')
print("\n✅ Model trained and saved as 'pose_lstm_model.h5'")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 60, 64)         │        26,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 60, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 134,598 (525.77 KB)

 Trainable params: 134,214 (524.27 KB)

 Non-trainable params: 384 (1.50 KB)


Starting Training...
Epoch 1/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 132ms/step - accuracy: 0.2558 - loss: 1.9495 - val_accuracy: 0.4419 - val_loss: 1.6376
Epoch 2/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.5698 - loss: 1.1758 - val_accuracy: 0.4419 - val_loss: 1.5458
Epoch 3/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.7151 - loss: 0.8944 - val_accuracy: 0.4884 - val_loss: 1.4724
Epoch 4/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.7965 - loss: 0.6610 - val_accuracy: 0.5116 - val_loss: 1.4099
Epoch 5/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.8314 - loss: 0.5504 - val_accuracy: 0.5116 - val_loss: 1.3594
Epoch 6/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.8256 - loss: 0.5436 - val_accuracy: 0.5349 - val_loss: 1.3134
Epoch 7/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.8721 - loss: 0.4071 - val_accuracy: 0.5116 - val_loss: 1.2585
Epoch 8/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.8779 - loss: 0.3732 - val_accuracy: 0.4


✅ Model trained and saved as 'pose_lstm_model.h5'
